# 📥 01 — Ingestion Bronze
Chargement du fichier JSON depuis S3 → `BRONZE.HOUSE_PRICE_STRUCTURED`

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Création des objets si absents (idempotent)
session.sql("CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB").collect()
for schema in ["RAW","BRONZE","SILVER","GOLD","ML"]:
    session.sql(f"CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.{schema}").collect()

session.sql("USE DATABASE HOUSE_PRICE_DB").collect()
session.sql("USE SCHEMA BRONZE").collect()
print("Session active :", session.get_current_database(), "/", session.get_current_schema())

In [ ]:
# Vérification du fichier dans le stage
session.sql("LIST @RAW.S3_HOUSE_STAGE").show()
session.sql("SELECT $1 FROM @RAW.S3_HOUSE_STAGE LIMIT 3").show(3)

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE BRONZE.HOUSE_PRICE_RAW (
    RAW_DATA VARIANT,
    _LOAD_FILENAME VARCHAR(200),
    _LOAD_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)""").collect()
result = session.sql("""
COPY INTO BRONZE.HOUSE_PRICE_RAW (RAW_DATA, _LOAD_FILENAME)
FROM (SELECT $1, METADATA$FILENAME FROM @RAW.S3_HOUSE_STAGE)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
ON_ERROR = 'CONTINUE'
""").collect()
print("COPY result:", result)

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE BRONZE.HOUSE_PRICE_STRUCTURED AS
SELECT
    RAW_DATA:price::NUMBER AS PRICE,
    RAW_DATA:area::NUMBER AS AREA,
    RAW_DATA:bedrooms::NUMBER AS BEDROOMS,
    RAW_DATA:bathrooms::NUMBER AS BATHROOMS,
    RAW_DATA:stories::NUMBER AS STORIES,
    RAW_DATA:mainroad::VARCHAR AS MAINROAD,
    RAW_DATA:guestroom::VARCHAR AS GUESTROOM,
    RAW_DATA:basement::VARCHAR AS BASEMENT,
    RAW_DATA:hotwaterheating::VARCHAR AS HOTWATERHEATING,
    RAW_DATA:airconditioning::VARCHAR AS AIRCONDITIONING,
    RAW_DATA:parking::NUMBER AS PARKING,
    RAW_DATA:prefarea::VARCHAR AS PREFAREA,
    RAW_DATA:furnishingstatus::VARCHAR AS FURNISHINGSTATUS,
    _LOAD_FILENAME,
    _LOAD_TIMESTAMP
FROM BRONZE.HOUSE_PRICE_RAW
WHERE RAW_DATA IS NOT NULL""").collect()
print("HOUSE_PRICE_STRUCTURED créée")

In [ ]:
qc = session.sql("""
SELECT
    COUNT(*) AS total_lignes,
    COUNT_IF(PRICE IS NULL) AS price_nulls,
    COUNT_IF(AREA IS NULL) AS area_nulls,
    MIN(PRICE) AS price_min, MAX(PRICE) AS price_max,
    ROUND(AVG(PRICE),0) AS price_avg,
    COUNT(DISTINCT FURNISHINGSTATUS) AS furnishing_card
FROM BRONZE.HOUSE_PRICE_STRUCTURED""")
qc.show()
session.sql("SELECT * FROM BRONZE.HOUSE_PRICE_STRUCTURED LIMIT 5").show()